Берем 2 разные предобученнын модели, желательно Bert + что-то еще. Используем их для получения эмбедингов текста. В Bert как эмбединг текста берем CLS токен. Полученные эмбединги используем для классификации внешним классификатором - возмите 2 классификатора.

Если получится: дообучаем для задачи классификации  одну из моделей, на одном из датасетов

С BERT у вас 3 части у задания. Они отличны по сути:

1. Feature extraction (первая частт) — берём BERT, достаём эмбеддинги, обучаем классификатор (LogReg, SVM и т.д.). Плюс: быстро, мало данных нужно, видно что делают классификаторы. По сути используется тот классификатри, который есть, а BERT чтобы вытащить векторное представление текста. Для этого кстати тоже есть пара вариантов. Вы берете CLS токен, про которыы много говорили - в нем обощенное представление для дркумента. Если бы не было его, то из BERT вы доставали бы эмбединги слов текста, и дальше бы думали как из эмбедингов слов текста собрать вектор представляющий тексты, в том числе с учетом того, что эти вектора должны иметь одинаковую длину.
❗️Это часть 1. - CLS из обычного BERT для представления текста -> классификатор

2. Fine-tuning — дообучаем саму BERT на конкретном датасете. Плюс: обычно точнее, модель адаптируется к задаче. Минус: дольше, хотим GPU.
После этого fine-tuned модель можно использовать для:

Получения CLS:
❗️Это часть 2: CLS из дообученного BERT -> тот же классиыикатор, что и в задании 1.
Сравнить полученные результаты.

или

❗️это часть 3 задания: использовать fine-tuned BERT сразу для классификации.
Сравнить с 1 и 2.

Да, для обоих датасетов с которыми мы работаем, есть готовые fine-tuned модели на HuggingFace. Попробуйте их. Если получилось сделать свои fine-tuned модели, сравните.

Emotion (6 классов):

• bhadresh-savani/bert-base-uncased-emotion — BERT-base, fine-tuned именно на dair-ai/emotion

• IsmaelMousa/bert-finetuned-emotion

• Panda0116/emotion-classification-model — DistilBERT

20 Newsgroups:

• rjac/bert-20news-classification
— DistilBERT

На что обратить внимание:

Если берем просто BERT - то оттуда берем только векторное представление для текста (CLS) или для слов (и дальше как-то из векторов слов строим вектор для текста).

А вот если fine-tuned модель, то опций больше. Когда BERT fine-tunят на классификацию, к BERT прикрепляют линейный слой (классификатор). Модель на входе — текст, на выходе — сразу предсказание класса. Этот классификатор обучался вместе с BERT.

Дальше два варианта использования fine-tuned модели:

1. Как есть — используем встроенный классификатор (тот, что обучался при fine-tuning). Просто pipeline: текст → модель → предсказание. Быстро, но нет выбора классификатора.

2. Вытаскиваем эмбеддинги — берём fine-tuned BERT без классификатора, достаём CLS, и обучаем свой SVM/LogReg/что угодно. Это даёт выбор и можно сравнить что лучше подходит.

Эмбеддинги из fine-tuned BERT будут другими, чем из обычного — модель уже "знает" про нашу задачу, векторы лучше разделены по классам. На них любой классификатор (даже простой SVM) может работать лучше

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from transformers import BitsAndBytesConfig
from sklearn.preprocessing import StandardScaler
from transformers import pipeline
from transformers import BertTokenizer, BertModel, RobertaTokenizer, RobertaModel, AutoTokenizer,AutoModel, AutoModelForSequenceClassification
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import BertForSequenceClassification, TrainingArguments, Trainer,EarlyStoppingCallback, AutoModelForSequenceClassification
from torch.utils.data import Dataset
from tqdm import tqdm
from datasets import load_dataset
from peft import PeftModel
from sklearn.datasets import fetch_20newsgroups
from sklearn.preprocessing import LabelEncoder, StandardScaler
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"{device}")

cuda


In [2]:
class CLSDataset(Dataset):
    def __init__(self, texts, labels, model, tokenizer, device, max_length=128):
        self.texts = texts
        self.labels = labels
        self.max_length = max_length
        self.device = device
        self.embeddings = self._get_cls_embeddings(texts, model, tokenizer)
    def _get_cls_embeddings(self, texts, model, tokenizer, batch_size=32):
        model.eval()
        embeddings = []
        for i in tqdm(range(0, len(texts), batch_size), desc=f"CLS embeddings ({model.__class__.__name__})"):
            batch_texts = texts[i:i+batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt'
            )
            input_ids = encoded['input_ids'].to(self.device)
            attention_mask = encoded['attention_mask'].to(self.device)
            with torch.no_grad():
                outputs = model(input_ids, attention_mask=attention_mask)
                cls_embeddings = outputs.last_hidden_state[:, 0, :]
                embeddings.append(cls_embeddings.cpu().numpy())
        return np.vstack(embeddings)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return torch.tensor(self.embeddings[idx], dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

def dataset_to_sklearn(dataset):
    embeddings_list = []
    labels_list = []
    for i in range(len(dataset)):
        emb, label = dataset[i]
        embeddings_list.append(emb.numpy())
        labels_list.append(label.numpy())
    return np.array(embeddings_list), np.array(labels_list)

In [3]:
if __name__ == '__main__':
  bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
  bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
  bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
  bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
  roberta_model = RobertaModel.from_pretrained('roberta-base').to(device)
  roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
  dt = load_dataset('emotion')
  train_texts = list(dt['train']['text'])
  train_labels = list(dt['train']['label'])
  test_texts = list(dt['test']['text'])
  test_labels = list(dt['test']['label'])
  X_train, X_val, y_train, y_val = train_test_split(train_texts, train_labels, test_size=0.2)
  encoder = LabelEncoder()
  y_train = encoder.fit_transform(y_train)
  y_val = encoder.transform(y_val)
  train_bert = CLSDataset(X_train, y_train, bert_model, bert_tokenizer, device)
  val_bert = CLSDataset(X_val, y_val, bert_model, bert_tokenizer, device)
  test_bert = CLSDataset(test_texts, test_labels, bert_model, bert_tokenizer, device)
  X_train_bert, y_train_bert = dataset_to_sklearn(train_bert)
  X_val_bert, y_val_bert = dataset_to_sklearn(val_bert)
  X_test_bert, y_test_bert = dataset_to_sklearn(test_bert)
  train_roberta = CLSDataset(X_train, y_train, roberta_model, roberta_tokenizer, device)
  val_roberta = CLSDataset(X_val, y_val, roberta_model, roberta_tokenizer, device)
  test_roberta = CLSDataset(test_texts, test_labels, roberta_model, roberta_tokenizer, device)
  X_train_roberta, y_train_roberta = dataset_to_sklearn(train_roberta)
  X_val_roberta, y_val_roberta = dataset_to_sklearn(val_roberta)
  X_test_roberta, y_test_roberta = dataset_to_sklearn(test_roberta)
  scaler_bert = StandardScaler()
  X_train_bert = scaler_bert.fit_transform(X_train_bert)
  X_val_bert = scaler_bert.transform(X_val_bert)
  X_test_bert = scaler_bert.transform(X_test_bert)

  scaler_roberta = StandardScaler()
  X_train_roberta = scaler_roberta.fit_transform(X_train_roberta)
  X_val_roberta = scaler_roberta.transform(X_val_roberta)
  X_test_roberta = scaler_roberta.transform(X_test_roberta)
  classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Linear SVM': LinearSVC(max_iter=2000, random_state=42, dual='auto')
  }
  results = {
    'BERT': {},
    'RoBERTa': {}
  }
  for name, clf in classifiers.items():
    print(f"\nTraining {name}. BERT..")
    clf.fit(X_train_bert, y_train_bert)
    y_pred_val = clf.predict(X_val_bert)
    y_pred_test = clf.predict(X_test_bert)
    f1_val = f1_score(y_val_bert, y_pred_val, average='macro')
    f1_test = f1_score(y_test_bert, y_pred_test, average='macro')

    results['BERT'][name] = {'val_f1': f1_val, 'test_f1': f1_test}

    print(f"  Validation F1-macro: {f1_val:.4f}")
    print(f"  Test F1-macro: {f1_test:.4f}")

  for name, clf in classifiers.items():
    print(f"\nTraining {name}.RoBERT..")
    clf.fit(X_train_roberta, y_train_roberta)

    y_pred_val = clf.predict(X_val_roberta)
    y_pred_test = clf.predict(X_test_roberta)

    f1_val = f1_score(y_val_roberta, y_pred_val, average='macro')

    results['RoBERTa'][name] = {'val_f1': f1_val, 'test_f1': f1_test}

    print(f"  Validation F1-macro: {f1_val:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

CLS embeddings (RobertaModel): 100%|██████████| 63/63 [00:06<00:00,  9.67it/s]



Training Logistic Regression. BERT..
  Validation F1-macro: 0.4489
  Test F1-macro: 0.4633

Training Linear SVM. BERT..
  Validation F1-macro: 0.4441
  Test F1-macro: 0.4740

Training Logistic Regression.RoBERT..
  Validation F1-macro: 0.5335

Training Linear SVM.RoBERT..
  Validation F1-macro: 0.5304


Linear SVM - 0.459 - лучшее по BERT

LogRegr - 0.569 - лучшее по RoBERT

In [4]:
if __name__ == '__main__':
  bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
  bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
  bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
  bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
  roberta_model = RobertaModel.from_pretrained('roberta-base').to(device)
  roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
  categories = [
    'comp.sys.ibm.pc.hardware',
    'comp.sys.mac.hardware',
    'comp.graphics',
    'comp.windows.x'
  ]

  newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
  )
  all_texts = newsgroups.data
  all_labels = newsgroups.target
  X_train_full, X_test, y_train_full, y_test = train_test_split(
    all_texts, all_labels,
    test_size=0.2,
    random_state=42,
    stratify=all_labels
  )
  X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
  )
  encoder = LabelEncoder()
  y_train = encoder.fit_transform(y_train)
  y_val = encoder.transform(y_val)
  train_bert = CLSDataset(X_train, y_train, bert_model, bert_tokenizer, device)
  val_bert = CLSDataset(X_val, y_val, bert_model, bert_tokenizer, device)
  test_bert = CLSDataset(test_texts, test_labels, bert_model, bert_tokenizer, device)
  X_train_bert, y_train_bert = dataset_to_sklearn(train_bert)
  X_val_bert, y_val_bert = dataset_to_sklearn(val_bert)
  X_test_bert, y_test_bert = dataset_to_sklearn(test_bert)
  train_roberta = CLSDataset(X_train, y_train, roberta_model, roberta_tokenizer, device)
  val_roberta = CLSDataset(X_val, y_val, roberta_model, roberta_tokenizer, device)
  test_roberta = CLSDataset(test_texts, test_labels, roberta_model, roberta_tokenizer, device)
  X_train_roberta, y_train_roberta = dataset_to_sklearn(train_roberta)
  X_val_roberta, y_val_roberta = dataset_to_sklearn(val_roberta)
  X_test_roberta, y_test_roberta = dataset_to_sklearn(test_roberta)
  scaler_bert = StandardScaler()
  X_train_bert = scaler_bert.fit_transform(X_train_bert)
  X_val_bert = scaler_bert.transform(X_val_bert)
  X_test_bert = scaler_bert.transform(X_test_bert)

  scaler_roberta = StandardScaler()
  X_train_roberta = scaler_roberta.fit_transform(X_train_roberta)
  X_val_roberta = scaler_roberta.transform(X_val_roberta)
  X_test_roberta = scaler_roberta.transform(X_test_roberta)
  classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Linear SVM': LinearSVC(max_iter=2000, random_state=42, dual='auto')
  }
  results = {
    'BERT': {},
    'RoBERTa': {}
  }
  for name, clf in classifiers.items():
    print(f"\nTraining {name}. BERT..")
    clf.fit(X_train_bert, y_train_bert)
    y_pred_val = clf.predict(X_val_bert)
    y_pred_test = clf.predict(X_test_bert)
    f1_val = f1_score(y_val_bert, y_pred_val, average='macro')
    f1_test = f1_score(y_test_bert, y_pred_test, average='macro')

    results['BERT'][name] = {'val_f1': f1_val, 'test_f1': f1_test}

    print(f"  Validation F1-macro: {f1_val:.4f}")

  for name, clf in classifiers.items():
    print(f"\nTraining {name}.RoBERT..")
    clf.fit(X_train_roberta, y_train_roberta)

    y_pred_val = clf.predict(X_val_roberta)

    f1_val = f1_score(y_val_roberta, y_pred_val, average='macro')

    results['RoBERTa'][name] = {'val_f1': f1_val, 'test_f1': f1_test}

    print(f"  Validation F1-macro: {f1_val:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
CLS embeddings (RobertaModel): 100%|██████████| 63/63 [00:06<00:00,  9.44it/s]



Training Logistic Regression. BERT..
  Validation F1-macro: 0.6167

Training Linear SVM. BERT..
  Validation F1-macro: 0.6120

Training Logistic Regression.RoBERT..
  Validation F1-macro: 0.6644

Training Linear SVM.RoBERT..
  Validation F1-macro: 0.6386


LogRegr - 61.6 лучшее по BERT

LogRegr - 66.4 лучшее по RoBERT

Fine-Tune

BERT EMBEDS -> CLASSIFIER

In [5]:
def extract_cls_embeddings(texts, model, tokenizer, device, batch_size=32, max_length=64):
    model.eval()
    embeddings = []
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
        print(f"  pad_token установлен: '{tokenizer.pad_token}'")

    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting CLS embeddings"):
        batch_texts = texts[i:i+batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)

    return np.vstack(embeddings)

In [6]:
import os

In [9]:
if __name__ == '__main__':
    dt = load_dataset('emotion')
    train_texts = list(dt['train']['text'])
    train_labels = list(dt['train']['label'])
    test_texts = list(dt['test']['text'])
    test_labels = list(dt['test']['label'])

    X_train, X_val, y_train, y_val = train_test_split(
        train_texts, train_labels, test_size=0.2, random_state=42, stratify=train_labels
    )

    encoder = LabelEncoder()
    y_train_encoded = encoder.fit_transform(y_train)
    y_val_encoded = encoder.transform(y_val)
    test_labels_encoded = encoder.transform(test_labels)

    num_labels = len(encoder.classes_)
    label_names = encoder.classes_
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=num_labels
    )
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
        model.resize_token_embeddings(len(tokenizer))

    model = model.to(device)
    max_length = 64

    train_encodings = tokenizer(
        X_train,
        truncation=True,
        padding='max_length',
        max_length=max_length,
    )
    val_encodings = tokenizer(
        X_val,
        truncation=True,
        padding='max_length',
        max_length=max_length,
    )
    test_encodings = tokenizer(
        test_texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
    )

    from datasets import Dataset

    train_dataset = Dataset.from_dict({
        'input_ids': train_encodings['input_ids'],
        'attention_mask': train_encodings['attention_mask'],
        'label': y_train_encoded
    })

    val_dataset = Dataset.from_dict({
        'input_ids': val_encodings['input_ids'],
        'attention_mask': val_encodings['attention_mask'],
        'label': y_val_encoded
    })

    test_dataset = Dataset.from_dict({
        'input_ids': test_encodings['input_ids'],
        'attention_mask': test_encodings['attention_mask'],
        'label': test_labels_encoded
    })

    train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {
            'f1_macro': f1_score(labels, predictions, average='macro')
        }

    training_args = TrainingArguments(
        output_dir='./bert_finetuned_results',
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=100,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        save_total_limit=2,
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    test_predictions = trainer.predict(test_dataset)
    test_preds = np.argmax(test_predictions.predictions, axis=1)
    test_f1_bert = f1_score(test_labels_encoded, test_preds, average='macro')
    print(f"\nBERT Fine-tuned (прямая классификация):")
    print(f"  F1-macro: {test_f1_bert:.4f}")

    model_save_path = './bert_emotion_finetuned'
    model.save_pretrained(model_save_path)
    tokenizer.save_pretrained(model_save_path)

    bert_model_for_emb = AutoModel.from_pretrained(model_save_path)
    bert_model_for_emb = bert_model_for_emb.to(device)
    bert_model_for_emb.eval()
    tokenizer_for_emb = AutoTokenizer.from_pretrained(model_save_path)
    X_train_emb = extract_cls_embeddings(X_train, bert_model_for_emb, tokenizer_for_emb, device, max_length=64)
    X_val_emb = extract_cls_embeddings(X_val, bert_model_for_emb, tokenizer_for_emb, device, max_length=64)
    X_test_emb = extract_cls_embeddings(test_texts, bert_model_for_emb, tokenizer_for_emb, device, max_length=64)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_emb)
    X_val_scaled = scaler.transform(X_val_emb)
    X_test_scaled = scaler.transform(X_test_emb)

    best_c = 1.0
    best_f1 = 0

    for c in [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
        lr = LogisticRegression(max_iter=1000, C=c, random_state=42, multi_class='ovr')
        lr.fit(X_train_scaled, y_train_encoded)
        y_val_pred = lr.predict(X_val_scaled)
        f1 = f1_score(y_val_encoded, y_val_pred, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_c = c
    final_lr = LogisticRegression(max_iter=1000, C=best_c, random_state=42, multi_class='ovr')
    final_lr.fit(X_train_scaled, y_train_encoded)
    y_test_pred_lr = final_lr.predict(X_test_scaled)
    test_f1_lr = f1_score(test_labels_encoded, y_test_pred_lr, average='macro')

    print(f"\n Logistic Regression (на эмбеддингах fine-tuned BERT):")
    print(f"  F1-macro: {test_f1_lr:.4f}")
    lin = LinearSVC(max_iter=2000, random_state=42, dual='auto')
    lin.fit(X_train_scaled, y_train_encoded)
    y_test_pred_svc = lin.predict(X_test_scaled)
    test_f1_svc = f1_score(test_labels_encoded, y_test_pred_svc, average='macro')
    print(f"\n Linear SVC (на эмбеддингах fine-tuned BERT):")
    print(f"  F1-macro: {test_f1_svc:.4f}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro
1,0.353148,0.221721,0.898569
2,0.318166,0.197137,0.895133
3,0.202084,0.185169,0.899754


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


BERT Fine-tuned (прямая классификация):
  F1-macro: 0.8822


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./bert_emotion_finetuned
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Extracting CLS embeddings: 100%|██████████| 63/63 [00:06<00:00,  9.50it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/


 Logistic Regression (на эмбеддингах fine-tuned BERT):
  F1-macro: 0.8804

 Linear SVC (на эмбеддингах fine-tuned BERT):
  F1-macro: 0.8804


BERT CLASSIFIER

In [8]:
if __name__ == '__main__':
    dt = load_dataset('emotion')
    test_texts = list(dt['test']['text'])
    test_labels = list(dt['test']['label'])

    id_to_emotion = {
        0: 'sadness',
        1: 'joy',
        2: 'love',
        3: 'anger',
        4: 'fear',
        5: 'surprise'
    }
    emotion_to_id = {v: k for k, v in id_to_emotion.items()}
    classifier_pipeline = pipeline(
        "text-classification",
        model='bhadresh-savani/bert-base-uncased-emotion'
    )
    sample_size = 500
    predictions = []

    for text in tqdm(test_texts[:sample_size], desc="Predicting"):
        result = classifier_pipeline(text)[0]
        pred_label = result['label']
        if pred_label.startswith('LABEL_'):
            pred_id = int(pred_label.split('_')[1])
        else:
            pred_id = emotion_to_id.get(pred_label, -1)

        if pred_id == -1:
            print(f"Предупреждение: неизвестная метка '{pred_label}'")
            continue

        predictions.append(pred_id)
    f1_direct = f1_score(test_labels[:len(predictions)], predictions, average='macro')
    print(f"\nРезультаты:")
    print(f"  F1-macro: {f1_direct:.4f}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bhadresh-savani/bert-base-uncased-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Predicting: 100%|██████████| 500/500 [00:04<00:00, 102.45it/s]


Результаты:
  F1-macro: 0.8826
